# MovieNet-318 Scene Segmentation — Clustering Baselines

Embedding-based scene boundary detection on the **MovieNet-318** benchmark using subtitle, visual (CLIP), and multimodal features.

**T4-friendly setup:** clone this repo, install dependencies, then download precomputed embeddings from [asmith06/scene-segmentation-embeddings](https://huggingface.co/datasets/asmith06/scene-segmentation-embeddings) (~3 GB). Keyframes (~50 GB) are optional.

| Required in repo | Purpose |
|------------------|---------|
| `data/scene318/label318/` | Scene boundary labels |
| `data/scene318/meta/split318.json` | Train / val / test split |
| `data/scene318/shot_movie318/` | Shot timing for subtitle alignment |
| `data/subtitle/` | SRT subtitles (314/318 movies) |
| `data/embeddings/` | Precomputed embeddings (HF download) |
| `data/keyframes_240p/` | Optional — only if building CLIP from scratch |


## Environment setup

Clones this repository (Colab) or locates it locally, then defines paths under `data/`. Override with environment variables:

- `SCENE_SEG_REPO_URL` — git clone URL (Colab)
- `SCENE_SEG_REPO_DIR` — path to repo root (local Jupyter)

In [ ]:
# --- Environment setup (Colab or local) ---
import os
import subprocess
import sys
from pathlib import Path

# Set your fork URL after publishing, or clone this repo locally.
REPO_URL = os.environ.get(
    "SCENE_SEG_REPO_URL",
    "https://github.com/lwan1/movienet-scene-segmentation.git",
)
REPO_DIR = Path(os.environ.get("SCENE_SEG_REPO_DIR", "/content/movienet-scene-segmentation"))

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
elif not REPO_DIR.exists():
    # Local: assume notebook lives in repo root or notebooks/
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "data" / "scene318" / "meta" / "split318.json").exists():
            REPO_DIR = candidate
            break
    else:
        raise FileNotFoundError(
            "Could not find repo data/. Clone the repo or set SCENE_SEG_REPO_DIR."
        )

DATA_ROOT = REPO_DIR / "data"
KEYFRAME_ZIP = DATA_ROOT / "keyframes_240p.zip"
KEYFRAME_DIR = DATA_ROOT / "keyframes_240p"

print("REPO_DIR:", REPO_DIR.resolve())
print("DATA_ROOT:", DATA_ROOT.resolve())


## Install dependencies

Installs Python packages from `requirements.txt`, including PyTorch, transformers, and the Hugging Face CLI used for keyframe download.

In [ ]:
# Install dependencies (after REPO_DIR is set)
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)


## Download assets (T4-friendly)

**Recommended on Colab T4:** `DOWNLOAD_EMBEDDINGS = True`, `DOWNLOAD_KEYFRAMES = False`.

Precomputed embeddings (~3 GB) from [asmith06/scene-segmentation-embeddings](https://huggingface.co/datasets/asmith06/scene-segmentation-embeddings) mirror `data/embeddings/` (`visual/`, `subtitle/`, `multimodal/`).

Keyframes (~50 GB) are **optional** — only for building CLIP from scratch or visualization. Uses streaming extraction via `scripts/setup_keyframes.py` (no `extractall`).

In [ ]:
# --- T4-friendly asset toggles ---
DOWNLOAD_EMBEDDINGS = True   # ~3 GB HF download — recommended on Colab T4
DOWNLOAD_KEYFRAMES = False   # ~50 GB — only if encoding CLIP from scratch or visualization
KEYFRAME_SUBSET_ONLY = True  # when DOWNLOAD_KEYFRAMES=True, extract only movies in the current split/subset

import json
import os
import subprocess
import sys

HF_EMBEDDINGS_REPO = "asmith06/scene-segmentation-embeddings"
HF_KEYFRAMES_DATASET = "asmith06/movienet318-keyframes-240p"
HF_KEYFRAMES_FILE = "keyframes_240p.zip"


def keyframes_ready(movie_ids=None) -> bool:
    if not KEYFRAME_DIR.exists():
        return False
    if movie_ids:
        return all((KEYFRAME_DIR / mid).is_dir() for mid in movie_ids)
    return any(KEYFRAME_DIR.glob("tt*"))


def embeddings_ready(movie_ids, modalities=("visual", "subtitle", "multimodal")) -> bool:
    root = DATA_ROOT / "embeddings"
    for modality in modalities:
        mod_dir = root / modality
        if not mod_dir.exists():
            return False
        if not all((mod_dir / f"{mid}.npz").exists() for mid in movie_ids):
            return False
    return True


def download_embeddings_hf(movie_ids=None, modalities=("visual", "subtitle", "multimodal")):
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts" / "download_embeddings_hf.py"),
        "--local-dir",
        str(DATA_ROOT / "embeddings"),
        "--repo",
        HF_EMBEDDINGS_REPO,
        "--modalities",
        *modalities,
    ]
    if movie_ids:
        cmd.extend(["--movie-ids", *movie_ids])
    subprocess.run(cmd, check=True, cwd=REPO_DIR)


def download_and_extract_keyframes(movie_ids=None):
    os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if not KEYFRAME_ZIP.exists():
        print(f"Downloading {HF_KEYFRAMES_FILE} from {HF_KEYFRAMES_DATASET} (~50 GB)...")
        subprocess.run(
            [
                "hf",
                "download",
                HF_KEYFRAMES_DATASET,
                HF_KEYFRAMES_FILE,
                "--repo-type",
                "dataset",
                "--local-dir",
                str(DATA_ROOT),
            ],
            check=True,
        )
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts" / "setup_keyframes.py"),
        "--zip-path",
        str(KEYFRAME_ZIP),
        "--out-dir",
        str(KEYFRAME_DIR),
    ]
    if movie_ids:
        cmd.extend(["--movie-ids", *movie_ids])
    subprocess.run(cmd, check=True, cwd=REPO_DIR)


def ensure_assets(movie_ids):
    """Download embeddings and/or keyframes for the given movie IDs."""
    if DOWNLOAD_EMBEDDINGS and not embeddings_ready(movie_ids):
        print(f"Downloading embeddings for {len(movie_ids)} movies from HF...")
        subset = movie_ids if len(movie_ids) < 318 else None
        download_embeddings_hf(subset)

    kf_ids = movie_ids if KEYFRAME_SUBSET_ONLY else None
    if DOWNLOAD_KEYFRAMES and not keyframes_ready(kf_ids):
        print("Streaming keyframe extraction (T4-safe, no extractall)...")
        download_and_extract_keyframes(kf_ids)
    elif not DOWNLOAD_KEYFRAMES:
        print("Skipping keyframe download (T4-friendly path).")
    else:
        print("Keyframes already present:", KEYFRAME_DIR)

    if DOWNLOAD_EMBEDDINGS:
        print("Embeddings root:", DATA_ROOT / "embeddings")
    print("KEYFRAME_DIR ready:", keyframes_ready(kf_ids if KEYFRAME_SUBSET_ONLY else None))


## Configuration

Global settings for the clustering pipeline:

- **`LABEL_POLICY = "lgss"`** — follows the LGSS benchmark convention (Rao et al., CVPR 2020): a shot is a scene boundary if its raw label is `1` or `-1` (hard cut / montage marker).
- **`LIMIT_MOVIES`** — set to a small integer (e.g. 10) for a fast smoke test; `None` runs all 318 movies.

In [ ]:
import json
import re
import warnings
import zipfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from PIL import Image
from sklearn.metrics import average_precision_score, precision_recall_curve
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

LABEL_POLICY = "lgss"
DEFAULT_FPS = 24.0
LIMIT_MOVIES: Optional[int] = None  # set to 10 for a quick smoke test

SOTA_REFERENCE = pd.DataFrame([
    {"method": "BaSSL", "f1": 47.0, "ap": 57.4},
    {"method": "TranS4mer", "f1": 48.4, "ap": 60.8},
    {"method": "MEGA", "f1": 55.3, "ap": 58.6},
    {"method": "Scene-VLM", "f1": 62.1, "ap": 66.8},
])
print("Config loaded.")


## Dataset paths

Maps repo directories to MovieNet-318 assets. Writable outputs (embeddings, plots, CSV results) go to `outputs/` so git-tracked `data/` stays read-only.

In [ ]:
# --- Dataset paths (relative to repo) ---
SCENE318_DIR = DATA_ROOT / "scene318"
LABEL_DIR = SCENE318_DIR / "label318"
SHOT_DIR = SCENE318_DIR / "shot_movie318"
SPLIT_PATH = SCENE318_DIR / "meta" / "split318.json"
SUBTITLE_DIR = DATA_ROOT / "subtitle"

# Writable outputs (created outside git-tracked data/)
WORK_ROOT = REPO_DIR / "outputs"
EDA_OUT_DIR = WORK_ROOT / "eda"
EMB_ROOT = DATA_ROOT / "embeddings" if DOWNLOAD_EMBEDDINGS else WORK_ROOT / "embeddings"
RESULTS_DIR = WORK_ROOT / "results"
CKPT_DIR = WORK_ROOT / "checkpoints" / "bassl"

for d in (EDA_OUT_DIR, EMB_ROOT, RESULTS_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

def check_paths():
    required = {"label318": LABEL_DIR, "split318.json": SPLIT_PATH}
    optional = {
        "shot_movie318": SHOT_DIR,
        "subtitles": SUBTITLE_DIR,
        "keyframes": KEYFRAME_DIR,
    }
    print("Required paths:")
    for name, p in required.items():
        print(f"  {'OK' if p.exists() else 'MISSING'}  {name}: {p}")
    print("\nOptional paths:")
    for name, p in optional.items():
        print(f"  {'OK' if p.exists() else 'skip'}  {name}: {p}")

check_paths()


## Data loading & label parsing

Loads the official **train / val / test** split (`split318.json`) and per-movie label files.

Each line in `label318/ttXXXX.txt` is `shot_id raw_label`. Under the **LGSS policy**:

$$
y_i = \begin{cases} 1 & \text{if } \text{raw}_i \in \{1, -1\} \\ 0 & \text{otherwise} \end{cases}
$$

Shot timing files align subtitle cues to shots; keyframe paths resolve per-movie image folders.

In [ ]:
def load_split(split_path=SPLIT_PATH):
    with open(split_path, "r", encoding="utf-8") as f:
        split = json.load(f)
    return {k: list(split[k]) for k in ("train", "val", "test")}


def parse_label_file(label_path, policy=LABEL_POLICY):
    rows = []
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = re.split(r"\s+", line)
            if len(parts) < 2:
                continue
            shot_id, raw = int(parts[0]), int(parts[1])
            if policy == "lgss":
                binary = 1 if raw == -1 else raw
            else:
                binary = raw
            rows.append({"shot_id": shot_id, "raw_label": raw, "is_boundary": int(binary)})
    return rows


def parse_shot_movie318(shot_path):
    rows = []
    with open(shot_path, "r", encoding="utf-8") as f:
        for shot_idx, line in enumerate(f):
            nums = [int(x) for x in line.strip().split() if x.strip()]
            if len(nums) < 2:
                continue
            rows.append({
                "shot_id": shot_idx,
                "start_frame": nums[0],
                "end_frame": nums[1],
                "duration_frames": nums[1] - nums[0],
                "kf_indices": nums[2:5] if len(nums) >= 5 else [],
            })
    return rows


def parse_srt_timestamp(ts: str) -> float:
    ts = ts.strip().replace(",", ".")
    parts = ts.split(":")
    h, m, s = int(parts[0]), int(parts[1]), float(parts[2])
    return h * 3600 + m * 60 + s


def parse_srt_file(srt_path: Path):
    cues = []
    blocks = re.split(r"\n\s*\n", srt_path.read_text(encoding="utf-8", errors="ignore").strip())
    for block in blocks:
        lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
        if len(lines) < 2:
            continue
        if "-->" not in lines[1]:
            continue
        start_s, end_s = [x.strip() for x in lines[1].split("-->")]
        text = " ".join(lines[2:])
        cues.append({
            "start_sec": parse_srt_timestamp(start_s),
            "end_sec": parse_srt_timestamp(end_s),
            "text": text,
        })
    return cues


def load_movie_meta(movie_id, meta_dir=META_DIR):
    if meta_dir is None:
        return None
    for candidate in (meta_dir / f"{movie_id}.json", meta_dir / "meta" / f"{movie_id}.json"):
        if candidate.exists():
            with open(candidate, "r", encoding="utf-8") as f:
                return json.load(f)
    return None


def find_subtitle_file(movie_id, subtitle_dir=SUBTITLE_DIR):
    if subtitle_dir is None or not Path(subtitle_dir).exists():
        return None
    subtitle_dir = Path(subtitle_dir)
    for ext in (".srt", ".txt", ".ass", ".vtt"):
        p = subtitle_dir / f"{movie_id}{ext}"
        if p.exists():
            return p
    matches = list(subtitle_dir.rglob(f"{movie_id}.*"))
    return matches[0] if matches else None


def find_movie_keyframe_dir(movie_id, keyframe_root=KEYFRAME_DIR):
    if keyframe_root is None:
        return None
    keyframe_root = Path(keyframe_root)
    for candidate in (keyframe_root / movie_id, keyframe_root / "240P_frames" / movie_id):
        if candidate.exists():
            return candidate
    matches = list(keyframe_root.rglob(movie_id))
    return matches[0] if matches else None


def align_subtitles_to_shots(shots, cues, fps=DEFAULT_FPS):
    texts = []
    for shot in shots:
        t0 = shot["start_frame"] / fps
        t1 = shot["end_frame"] / fps
        chunk = [c["text"] for c in cues if c["end_sec"] >= t0 and c["start_sec"] <= t1]
        texts.append(" ".join(chunk).strip())
    return texts


def labels_to_scene_lengths(label_rows):
    lengths, current = [], 0
    for row in label_rows:
        current += 1
        if row["is_boundary"] == 1:
            lengths.append(current)
            current = 0
    if current > 0:
        lengths.append(current)
    return lengths


split = load_split()
ALL_MOVIE_IDS = split["train"] + split["val"] + split["test"]
MOVIE_TO_SPLIT = {mid: name for name, ids in split.items() for mid in ids}
if LIMIT_MOVIES is not None:
    ALL_MOVIE_IDS = ALL_MOVIE_IDS[:LIMIT_MOVIES]
print(f"Movies: {len(ALL_MOVIE_IDS)} | train/val/test = {len(split['train'])}/{len(split['val'])}/{len(split['test'])}")

ensure_assets(ALL_MOVIE_IDS)


## Exploratory label statistics

Summarizes shot counts, boundary rates, and modality coverage (subtitles / keyframes) per split. Scene length statistics are computed from consecutive non-boundary runs.

In [ ]:
movie_records, shot_records = [], []
for movie_id in tqdm(ALL_MOVIE_IDS, desc="Loading labels"):
    label_path = LABEL_DIR / f"{movie_id}.txt"
    if not label_path.exists():
        continue
    rows = parse_label_file(label_path)
    raw = [r["raw_label"] for r in rows]
    lgss = [r["is_boundary"] for r in rows]
    meta = load_movie_meta(movie_id) or {}
    sub_path = find_subtitle_file(movie_id)
    movie_records.append({
        "movie_id": movie_id,
        "split": MOVIE_TO_SPLIT[movie_id],
        "title": meta.get("title"),
        "num_shots": len(rows),
        "n_raw_1": sum(1 for x in raw if x == 1),
        "n_minus1": sum(1 for x in raw if x == -1),
        "n_lgss_boundary": sum(lgss),
        "boundary_rate_lgss": sum(lgss) / max(len(rows), 1),
        "avg_scene_len": float(np.mean(labels_to_scene_lengths(rows))) if rows else np.nan,
        "has_subtitles": sub_path is not None,
        "has_keyframes": find_movie_keyframe_dir(movie_id) is not None,
        "primary_genre": (meta.get("genres") or ["Unknown"])[0],
    })
    for i, r in enumerate(rows):
        shot_records.append({"movie_id": movie_id, "split": MOVIE_TO_SPLIT[movie_id], "shot_idx": i, **r})

df_movies = pd.DataFrame(movie_records)
df_shots = pd.DataFrame(shot_records)

n_total = len(df_shots)
n_raw1 = int((df_shots["raw_label"] == 1).sum())
n_m1 = int((df_shots["raw_label"] == -1).sum())
n_lgss = int((df_shots["is_boundary"] == 1).sum())
print(f"Shots: {n_total:,}")
print(f"Raw 1: {n_raw1:,} ({100*n_raw1/n_total:.2f}%) | -1: {n_m1:,} | LGSS pos: {n_lgss:,} ({100*n_lgss/n_total:.2f}%)")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
pd.Series({"Not boundary": n_total - n_lgss, "Boundary (LGSS)": n_lgss}).plot(kind="bar", ax=ax[0])
ax[0].set_title("LGSS label imbalance")
pd.Series({"0": int((df_shots.raw_label==0).sum()), "1": n_raw1, "-1": n_m1}).plot(kind="bar", ax=ax[1], color=["C0","C1","C3"])
ax[1].set_title("Raw labels")
plt.tight_layout(); plt.savefig(EDA_OUT_DIR / "label_imbalance.png", dpi=150); plt.show()

display(df_movies.groupby("split").agg(n=("movie_id","count"), mean_shots=("num_shots","mean"), subtitle_cov=("has_subtitles","mean"), kf_cov=("has_keyframes","mean")))
df_movies.to_csv(EDA_OUT_DIR / "movie_summary.csv", index=False)

## Boundary scores & evaluation metrics

**Adjacent-distance score** (cosine dissimilarity between consecutive L2-normalized embeddings):

$$
s_i = 1 - \langle \hat{x}_{i-1}, \hat{x}_i \rangle, \quad \hat{x}_i = \frac{x_i}{\|x_i\|}
$$

Predictions threshold $s_i \geq \tau$. Metrics:

- **F1** (per movie, macro-averaged): $\mathrm{F1} = \frac{2PR}{P+R}$
- **AP**: area under the precision–recall curve (`sklearn.average_precision_score`)

In [ ]:
def boundary_scores_from_embeddings(X: np.ndarray) -> np.ndarray:
    """Score for shot i (i>=1): dissimilarity between shot i-1 and i."""
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    n = len(Xn)
    scores = np.zeros(n, dtype=float)
    for i in range(1, n):
        scores[i] = 1.0 - float(np.dot(Xn[i - 1], Xn[i]))
    return scores


def metrics_from_scores(y_true: np.ndarray, scores: np.ndarray):
    y_true = y_true.astype(int)
    if y_true.sum() == 0:
        return {"f1": np.nan, "ap": np.nan, "threshold": np.nan}
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1s = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    if len(f1s) == 0:
        best_f1, best_t = 0.0, 0.5
    else:
        idx = int(np.nanargmax(f1s))
        best_f1, best_t = float(f1s[idx]), float(thresholds[idx])
    ap = float(average_precision_score(y_true, scores))
    return {"f1": best_f1, "ap": ap, "threshold": best_t}


def predict_from_threshold(scores: np.ndarray, threshold: float) -> np.ndarray:
    return (scores >= threshold).astype(int)


## Shot embeddings (subtitle, visual, multimodal)

Builds and caches per-shot embeddings under `outputs/embeddings/`:

| Modality | Encoder | Input |
|----------|---------|-------|
| **subtitle** | MiniLM-L6-v2 | Subtitle text aligned to each shot |
| **visual** | CLIP ViT-B/32 | Middle keyframe (`shot_XXXX_img_1.jpg`) |
| **multimodal** | Concatenation | $[\hat{x}^{\text{vis}} ; \hat{x}^{\text{sub}}]$, L2-normalized |

Subtitles are aligned by overlapping SRT cue intervals with each shot's frame range (24 fps).

In [ ]:
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
text_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

clip_model = clip_processor = None
hf_has_visual = (EMB_ROOT / "visual").exists() and any((EMB_ROOT / "visual").glob("*.npz"))
if not hf_has_visual and KEYFRAME_DIR is not None and any(df_movies["has_keyframes"].tolist()):
    from transformers import CLIPModel, CLIPProcessor
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip_model.eval()
    print("CLIP loaded on", device)
elif hf_has_visual:
    print("Using precomputed visual/multimodal embeddings from", EMB_ROOT)
else:
    print("No keyframes found — visual/multimodal arms will be skipped; subtitle-only still runs.")


def load_or_build_embeddings(movie_id: str, emb_root_path: Path = EMB_ROOT, label_dir_path: Path = LABEL_DIR):
    paths = {
        "subtitle": emb_root_path / "subtitle" / f"{movie_id}.npz",
        "visual": emb_root_path / "visual" / f"{movie_id}.npz",
        "multimodal": emb_root_path / "multimodal" / f"{movie_id}.npz",
    }
    cached = {k: np.load(p)["X"] for k, p in paths.items() if p.exists()}
    if "subtitle" in cached and (clip_model is None or ("visual" in cached and "multimodal" in cached)):
        return cached

    label_rows = parse_label_file(label_dir_path / f"{movie_id}.txt")
    n = len(label_rows)

    # Shot timing
    shots = []
    if SHOT_DIR is not None and (SHOT_DIR / f"{movie_id}.txt").exists():
        shots = parse_shot_movie318(SHOT_DIR / f"{movie_id}.txt")
    if len(shots) != n:
        shots = [{"shot_id": i, "start_frame": i * 100, "end_frame": (i + 1) * 100} for i in range(n)]

    # Subtitle embeddings
    sub_path = find_subtitle_file(movie_id)
    if sub_path is not None and len(shots) > 0:
        cues = parse_srt_file(sub_path)
        texts = align_subtitles_to_shots(shots, cues, fps=DEFAULT_FPS)
    else:
        texts = [""] * n
    sub_emb = text_model.encode(texts, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    np.savez(paths["subtitle"], X=sub_emb)

    # Visual embeddings
    vis_emb = None
    kf_dir = find_movie_keyframe_dir(movie_id)
    if clip_model is not None and kf_dir is not None:
        imgs = []
        for i in range(n):
            img_path = kf_dir / f"shot_{i:04d}_img_1.jpg"
            if not img_path.exists():
                img_path = kf_dir / f"shot_{i:04d}_img_0.jpg"
            if img_path.exists():
                imgs.append(Image.open(img_path).convert("RGB"))
            else:
                imgs.append(Image.new("RGB", (224, 224), color=(0, 0, 0)))
        feats = []
        batch_size = 32
        for start in range(0, n, batch_size):
            batch = imgs[start:start + batch_size]
            inputs = clip_processor(images=batch, return_tensors="pt").to(device)
            with torch.no_grad():
                f_output = clip_model.get_image_features(**inputs)
                # Extract tensor from BaseModelOutputWithPooling if necessary
                if hasattr(f_output, 'pooler_output'):
                    f = f_output.pooler_output
                else:
                    f = f_output # Assume it's already the tensor

                # Normalize the features
                f = f / f.norm(dim=-1, keepdim=True)
            feats.append(f.cpu().numpy())
        vis_emb = np.vstack(feats)
        np.savez(paths["visual"], X=vis_emb)

    # Multimodal
    if vis_emb is not None:
        mm = np.concatenate([vis_emb, sub_emb], axis=1)
        mm = mm / (np.linalg.norm(mm, axis=1, keepdims=True) + 1e-12)
        np.savez(paths["multimodal"], X=mm)

    out = {"subtitle": sub_emb}
    if vis_emb is not None:
        out["visual"] = vis_emb
        out["multimodal"] = mm
    return out


# Pre-build embeddings for val+test (and train if time permits)
embed_ids = split["val"] + split["test"]
if LIMIT_MOVIES is not None:
    embed_ids = embed_ids[:LIMIT_MOVIES]
for mid in tqdm(embed_ids, desc="Embedding val+test"):
    try:
        load_or_build_embeddings(mid, emb_root_path=EMB_ROOT, label_dir_path=LABEL_DIR)
    except Exception as e:
        # Re-raise the exception to stop execution as requested by the user
        raise e


## Segmentation methods

Three unsupervised segmenters operate on embedding sequences:

1. **Adjacent distance** — threshold $s_i$ from above.
2. **Constrained merge** — mark shot $i{-}1$ as a boundary when $s_i \geq \tau$; force the final shot to end a scene.
3. **OSG dynamic programming** — partition shots to minimize $\sum_i (1 - \langle \hat{x}_{i-1}, \hat{x}_i \rangle)$ plus $\lambda$ per boundary (Han & Wu, 2011-style optimal grouping). $O(n^2)$ per movie; disabled by default for speed.

In [ ]:
def constrained_merge_boundaries(X: np.ndarray, merge_threshold: float) -> np.ndarray:
    X = X.copy()
    X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    n = len(X)
    if n == 0:
        return np.array([])
    # cluster represented by mean embedding per contiguous segment
    seg_start = 0
    boundaries = np.zeros(n, dtype=int)
    for i in range(1, n):
        d = 1.0 - float(np.dot(X[i - 1], X[i]))
        if d >= merge_threshold:
            boundaries[i - 1] = 1  # shot i-1 ends a scene
    boundaries[-1] = 1  # last shot ends final scene
    return boundaries


def osg_dp_boundaries(X: np.ndarray, boundary_penalty: float = 0.35) -> np.ndarray:
    X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    n = len(X)
    if n == 0:
        return np.array([])

    def seg_cost(a, b):
        if b <= a:
            return 0.0
        c = 0.0
        for i in range(a + 1, b + 1):
            c += 1.0 - float(np.dot(X[i - 1], X[i]))
        return c

    dp = np.full(n, np.inf)
    prev = np.full(n, -1, dtype=int)
    for i in range(n):
        for j in range(i + 1):
            cost = (dp[j - 1] if j > 0 else 0.0) + seg_cost(j, i) + boundary_penalty
            if cost < dp[i]:
                dp[i] = cost
                prev[i] = j - 1
    boundaries = np.zeros(n, dtype=int)
    i = n - 1
    while i >= 0:
        boundaries[i] = 1
        i = prev[i]
    return boundaries


METHODS = ["adjacent_distance", "constrained_merge", "osg_dp"]

def available_modalities():
    mods = ["subtitle"]
    if clip_model is not None and KEYFRAME_DIR is not None:
        mods.extend(["visual", "multimodal"])
    return mods


## Hyperparameter tuning (validation split)

For each **modality × method**, grid-search thresholds (or OSG penalty $\lambda$) on the **validation** split and keep the setting with highest macro F1. Embeddings are cached once per movie to avoid redundant encoding.

In [ ]:
from sklearn.metrics import f1_score

# --- Tuning speed knobs ---
# OSG DP is O(n²) per movie and pure Python (GPU does not help). Skip by default.
TUNING_METHODS = ["adjacent_distance", "constrained_merge"]  # append "osg_dp" to re-enable
TUNE_THRESHOLDS = np.linspace(0.10, 0.90, 9)   # coarser grid (was 19); increase for final runs
OSG_PENALTIES = np.linspace(0.20, 0.80, 7)     # only used if "osg_dp" is in TUNING_METHODS


def build_tuning_cache(movie_ids, modality):
    """Load embeddings once per movie; reuse across all hyperparameter trials."""
    cache = []
    for mid in tqdm(movie_ids, desc=f"Cache {modality}"):
        embs = load_or_build_embeddings(mid, emb_root_path=EMB_ROOT, label_dir_path=LABEL_DIR)
        if modality not in embs:
            continue
        X = embs[modality]
        y = np.array(
            [r["is_boundary"] for r in parse_label_file(LABEL_DIR / f"{mid}.txt")],
            dtype=int,
        )
        Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
        cache.append({"movie_id": mid, "y": y, "X": Xn, "scores": boundary_scores_from_embeddings(X)})
    return cache


def tune_on_split(cache, modality, method):
    if not cache:
        return {"f1": -1.0, "method": method, "modality": modality}
    best = {"f1": -1.0}
    if method == "adjacent_distance":
        for t in TUNE_THRESHOLDS:
            f1s = [
                f1_score(c["y"], predict_from_threshold(c["scores"], t), zero_division=0)
                for c in cache
            ]
            mean_f1 = float(np.mean(f1s))
            if mean_f1 > best["f1"]:
                best = {"f1": mean_f1, "threshold": float(t)}
    elif method == "constrained_merge":
        for t in TUNE_THRESHOLDS:
            f1s = [
                f1_score(c["y"], constrained_merge_boundaries(c["X"], merge_threshold=t), zero_division=0)
                for c in cache
            ]
            mean_f1 = float(np.mean(f1s))
            if mean_f1 > best["f1"]:
                best = {"f1": mean_f1, "merge_threshold": float(t)}
    elif method == "osg_dp":
        for p in OSG_PENALTIES:
            f1s = [
                f1_score(c["y"], osg_dp_boundaries(c["X"], boundary_penalty=p), zero_division=0)
                for c in cache
            ]
            mean_f1 = float(np.mean(f1s))
            if mean_f1 > best["f1"]:
                best = {"f1": mean_f1, "osg_penalty": float(p)}
    else:
        raise ValueError(f"Unknown method: {method}")
    best["method"] = method
    best["modality"] = modality
    return best


val_ids = [m for m in split["val"] if LIMIT_MOVIES is None or m in ALL_MOVIE_IDS]
MODALITIES = available_modalities()
print("Modalities:", MODALITIES)
print("Tuning methods:", TUNING_METHODS, "| grid size:", len(TUNE_THRESHOLDS))

tuning_rows = []
for modality in MODALITIES:
    cache = build_tuning_cache(val_ids, modality)
    print(f"{modality}: cached {len(cache)} val movies")
    for method in TUNING_METHODS:
        row = tune_on_split(cache, modality, method)
        if row.get("f1", -1) >= 0:
            tuning_rows.append(row)
            print(row)

df_tune = pd.DataFrame(tuning_rows)
df_tune.to_csv(RESULTS_DIR / "val_tuning.csv", index=False)
display(df_tune.sort_values("f1", ascending=False))


## Test evaluation

Applies the best validation hyperparameters to the **test** split (no re-tuning). Results are saved to `outputs/results/test_results.csv` and compared against published MovieNet-318 reference numbers.

In [ ]:
def eval_test_with_params(row):
    mids = split["test"]
    if LIMIT_MOVIES is not None:
        mids = [m for m in mids if m in ALL_MOVIE_IDS]
    f1s, aps = [], []
    for mid in mids:
        embs = load_or_build_embeddings(mid)
        mod = row["modality"]
        if mod not in embs:
            continue
        y = np.array([r["is_boundary"] for r in parse_label_file(LABEL_DIR / f"{mid}.txt")])
        X = embs[mod]
        scores = boundary_scores_from_embeddings(X)
        method = row["method"]
        if method == "adjacent_distance":
            pred = predict_from_threshold(scores, row["threshold"])
        elif method == "constrained_merge":
            pred = constrained_merge_boundaries(X, merge_threshold=row["merge_threshold"])
        else:
            pred = osg_dp_boundaries(X, boundary_penalty=row["osg_penalty"])
        f1s.append(f1_score(y, pred, zero_division=0))
        aps.append(average_precision_score(y, scores))
    return {"test_f1": float(np.mean(f1s)) if f1s else np.nan, "test_ap": float(np.mean(aps)) if aps else np.nan, "n_movies": len(f1s)}


test_rows = []
for _, row in df_tune.iterrows():
    metrics = eval_test_with_params(row)
    test_rows.append({**row.to_dict(), **metrics})

df_results = pd.DataFrame(test_rows).sort_values("test_f1", ascending=False)
df_results.to_csv(RESULTS_DIR / "test_results.csv", index=False)

print("\n=== Our methods (test) ===")
display(df_results[["modality", "method", "test_f1", "test_ap", "n_movies"]])

print("\n=== Published reference (MovieNet-318, Scene-VLM Table 2) ===")
display(SOTA_REFERENCE)

best = df_results.iloc[0]
print(f"\nBest clustering config: {best['modality']} + {best['method']} | test F1={best['test_f1']:.3f}, AP={best['test_ap']:.3f}")


## Failure analysis (qualitative)

For ground-truth boundaries in sample test movies, compares visual vs. subtitle transition strength ($1 - \cos$) to identify dialogue-driven vs. visually driven boundary errors and montage markers (raw label $-1$).

In [ ]:
def failure_analysis(movie_id, modality="subtitle"):
    embs = load_or_build_embeddings(movie_id)
    if modality not in embs:
        return None
    X = embs[modality]
    rows = parse_label_file(LABEL_DIR / f"{movie_id}.txt")
    y = np.array([r["is_boundary"] for r in rows])
    visX = embs.get("visual")
    subX = embs.get("subtitle")
    records = []
    for i in range(1, len(rows)):
        if y[i] != 1:
            continue
        rec = {"movie_id": movie_id, "shot_idx": i, "raw_label": rows[i]["raw_label"]}
        if visX is not None:
            rec["visual_dist"] = 1 - float(np.dot(visX[i-1], visX[i]))
        if subX is not None:
            rec["subtitle_dist"] = 1 - float(np.dot(subX[i-1], subX[i]))
        records.append(rec)
    return records

fail_rows = []
for mid in split["test"][:10]:
    fail_rows.extend(failure_analysis(mid) or [])

if fail_rows:
    df_fail = pd.DataFrame(fail_rows)
    display(df_fail.head(10))
    if "visual_dist" in df_fail and "subtitle_dist" in df_fail:
        df_fail["dialogue_driven"] = (df_fail["subtitle_dist"] > df_fail["visual_dist"])
        print("Dialogue-driven boundary fraction:", df_fail["dialogue_driven"].mean())
    print("Montage proxy (raw -1 at boundary):", (df_fail.raw_label == -1).mean())
else:
    print("No failure-analysis rows (check embeddings).")
